In [1]:
# ================================================================
# 🎤 AI INTERVIEW ANALYZER USING TEXT AND SPEECH
# Google Colab - Complete Single Cell Application
# ================================================================

# Install libraries
!pip -q install transformers torch sentencepiece librosa soundfile SpeechRecognition

import os
import re
import math
import librosa
import soundfile as sf
import speech_recognition as sr
from collections import Counter
from transformers import pipeline

print("=" * 75)
print("             🎤 AI INTERVIEW ANALYZER")
print("                 TEXT + SPEECH ANALYSIS")
print("=" * 75)

# ================================================================
# 1. LOAD AI MODELS
# ================================================================

print("\n⏳ Loading AI models...")

# Sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Emotion model
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=1
)

print("✅ AI models loaded successfully!")


# ================================================================
# 2. CHOOSE INPUT TYPE
# ================================================================

print("\n" + "=" * 75)
print("SELECT INTERVIEW INPUT")
print("=" * 75)

print("""
1. Text Answer
2. Speech / Audio Answer
""")

choice = input("Enter 1 or 2: ").strip()


# ================================================================
# 3. TEXT INPUT
# ================================================================

if choice == "1":

    print("\n" + "-" * 75)

    question = input(
        "Enter the interview question:\n> "
    )

    print("\nEnter your interview answer:")

    answer = input("> ")

    audio_duration = None
    speech_rate = None


# ================================================================
# 4. SPEECH INPUT
# ================================================================

elif choice == "2":

    print("\n📁 Upload your interview audio file.")
    print("Supported examples: WAV, MP3, M4A")

    from google.colab import files

    uploaded = files.upload()

    if len(uploaded) == 0:
        print("❌ No audio file uploaded.")
        raise SystemExit

    audio_file = list(uploaded.keys())[0]

    print("\n🎧 Audio file:", audio_file)

    # ------------------------------------------------------------
    # Get audio duration
    # ------------------------------------------------------------

    try:

        audio_data, sample_rate = librosa.load(
            audio_file,
            sr=None
        )

        audio_duration = len(audio_data) / sample_rate

    except:

        audio_duration = None


    # ------------------------------------------------------------
    # Speech to Text
    # ------------------------------------------------------------

    print("\n🎤 Converting speech to text...")

    recognizer = sr.Recognizer()

    try:

        # Convert audio to WAV if required
        wav_file = "interview_audio.wav"

        audio_data, sample_rate = librosa.load(
            audio_file,
            sr=16000
        )

        sf.write(
            wav_file,
            audio_data,
            16000
        )

        with sr.AudioFile(wav_file) as source:

            audio = recognizer.record(source)

        answer = recognizer.recognize_google(audio)

        print("\n✅ Speech successfully converted to text.")

    except Exception as e:

        print("\n❌ Speech recognition failed.")
        print("Error:", e)

        raise SystemExit


    question = input(
        "\nEnter the interview question:\n> "
    )

else:

    print("\n❌ Invalid choice.")
    raise SystemExit


# ================================================================
# 5. BASIC TEXT VALIDATION
# ================================================================

if len(answer.strip()) < 10:

    print("\n❌ Please provide a longer interview answer.")
    raise SystemExit


# ================================================================
# 6. TEXT CLEANING
# ================================================================

clean_answer = re.sub(
    r"\s+",
    " ",
    answer
).strip()

lower_answer = clean_answer.lower()

print("\n⏳ Analyzing interview answer...")


# ================================================================
# 7. SENTIMENT ANALYSIS
# ================================================================

sentiment_result = sentiment_model(
    clean_answer[:512]
)[0]

sentiment_label = sentiment_result["label"]

sentiment_confidence = (
    sentiment_result["score"] * 100
)

if sentiment_label == "POSITIVE":

    sentiment = "Positive 😊"

else:

    sentiment = "Negative 😟"


# ================================================================
# 8. EMOTION ANALYSIS
# ================================================================

emotion_result = emotion_model(
    clean_answer[:512]
)[0][0]

emotion_label = emotion_result["label"]

emotion_confidence = (
    emotion_result["score"] * 100
)

emotion_names = {

    "anger": "Angry 😠",
    "disgust": "Disgusted 🤢",
    "fear": "Fearful 😨",
    "joy": "Confident / Happy 😊",
    "neutral": "Neutral 😐",
    "sadness": "Sad 😢",
    "surprise": "Surprised 😮"

}

emotion = emotion_names.get(
    emotion_label.lower(),
    emotion_label.capitalize()
)


# ================================================================
# 9. TEXT STATISTICS
# ================================================================

words = re.findall(
    r"\b[a-zA-Z]+\b",
    lower_answer
)

word_count = len(words)

characters = len(clean_answer)

sentences = re.findall(
    r"[.!?]+",
    clean_answer
)

sentence_count = max(
    1,
    len(sentences)
)

average_sentence_length = (
    word_count / sentence_count
)


# ================================================================
# 10. FILLER WORD DETECTION
# ================================================================

filler_words = {

    "um": 0,
    "uh": 0,
    "like": 0,
    "actually": 0,
    "basically": 0,
    "you know": 0,
    "i mean": 0,
    "sort of": 0,
    "kind of": 0,
    "hmm": 0

}

for filler in filler_words:

    filler_words[filler] = lower_answer.count(
        filler
    )

total_fillers = sum(
    filler_words.values()
)


# ================================================================
# 11. TECHNICAL KEYWORD DETECTION
# ================================================================

technical_keywords = [

    "python",
    "java",
    "javascript",
    "react",
    "html",
    "css",
    "sql",
    "mysql",
    "database",
    "api",
    "machine learning",
    "artificial intelligence",
    "ai",
    "data science",
    "cloud",
    "aws",
    "azure",
    "docker",
    "git",
    "github",
    "algorithm",
    "data structure",
    "oop",
    "object oriented",
    "testing",
    "debugging",
    "frontend",
    "backend",
    "full stack",
    "flask",
    "fastapi",
    "django"

]

detected_technical = []

for keyword in technical_keywords:

    if keyword in lower_answer:

        detected_technical.append(
            keyword
        )


# ================================================================
# 12. SOFT SKILLS DETECTION
# ================================================================

soft_skills = [

    "teamwork",
    "team",
    "communication",
    "leadership",
    "problem solving",
    "problem-solving",
    "adaptability",
    "time management",
    "creative",
    "responsibility",
    "collaboration",
    "learning",
    "initiative",
    "decision making",
    "decision-making"

]

detected_soft_skills = []

for skill in soft_skills:

    if skill in lower_answer:

        detected_soft_skills.append(
            skill
        )


# ================================================================
# 13. EXPERIENCE / PROJECT DETECTION
# ================================================================

experience_keywords = [

    "project",
    "internship",
    "intern",
    "experience",
    "developed",
    "created",
    "built",
    "implemented",
    "worked",
    "application",
    "system",
    "hackathon"

]

experience_count = 0

for word in experience_keywords:

    if word in lower_answer:

        experience_count += 1


# ================================================================
# 14. ANSWER STRUCTURE ANALYSIS
# ================================================================

structure_keywords = {

    "Situation": [
        "when",
        "during",
        "in my project",
        "in my internship"
    ],

    "Action": [
        "developed",
        "implemented",
        "created",
        "designed",
        "worked",
        "used"
    ],

    "Result": [
        "result",
        "improved",
        "increased",
        "reduced",
        "achieved",
        "successfully"
    ]

}

structure_score = 0

detected_structure = []

for section, keywords in structure_keywords.items():

    found = False

    for keyword in keywords:

        if keyword in lower_answer:

            found = True
            break

    if found:

        structure_score += 1
        detected_structure.append(
            section
        )


# ================================================================
# 15. CONFIDENCE INDICATORS
# ================================================================

confidence_words = [

    "confident",
    "successfully",
    "achieved",
    "experienced",
    "strong",
    "skilled",
    "comfortable",
    "capable",
    "learned",
    "solved"

]

confidence_count = 0

for word in confidence_words:

    if word in lower_answer:

        confidence_count += 1


# ================================================================
# 16. NEGATIVE / UNCERTAIN INDICATORS
# ================================================================

uncertain_words = [

    "maybe",
    "perhaps",
    "i don't know",
    "not sure",
    "i think",
    "probably",
    "cannot",
    "can't",
    "difficult",
    "confused"

]

uncertainty_count = 0

for word in uncertain_words:

    if word in lower_answer:

        uncertainty_count += 1


# ================================================================
# 17. RELEVANCE ANALYSIS
# ================================================================

question_words = set(
    re.findall(
        r"\b[a-zA-Z]{4,}\b",
        question.lower()
    )
)

answer_words = set(words)

common_words = (
    question_words.intersection(
        answer_words
    )
)

if len(question_words) > 0:

    relevance_score = (
        len(common_words)
        /
        len(question_words)
    ) * 100

else:

    relevance_score = 70


relevance_score = min(
    100,
    relevance_score
)


# ================================================================
# 18. SPEECH RATE
# ================================================================

if choice == "2" and audio_duration:

    speech_rate = (
        word_count /
        (audio_duration / 60)
    )

else:

    speech_rate = None


# ================================================================
# 19. COMMUNICATION SCORE
# ================================================================

communication_score = 50

# Good answer length
if word_count >= 50:
    communication_score += 15

elif word_count >= 30:
    communication_score += 10

elif word_count < 15:
    communication_score -= 15


# Sentence structure
if average_sentence_length >= 8:
    communication_score += 10


# Filler words
if total_fillers == 0:

    communication_score += 15

elif total_fillers <= 2:

    communication_score += 8

elif total_fillers >= 5:

    communication_score -= 15


# Uncertainty
communication_score -= (
    uncertainty_count * 5
)

communication_score = max(
    0,
    min(100, communication_score)
)


# ================================================================
# 20. TECHNICAL SCORE
# ================================================================

technical_score = min(
    100,
    len(detected_technical) * 10
)

if technical_score == 0:

    technical_score = 30


# ================================================================
# 21. SOFT SKILL SCORE
# ================================================================

soft_skill_score = min(
    100,
    len(detected_soft_skills) * 15
)

if soft_skill_score == 0:

    soft_skill_score = 40


# ================================================================
# 22. CONFIDENCE SCORE
# ================================================================

confidence_score = 60

confidence_score += (
    confidence_count * 7
)

confidence_score -= (
    uncertainty_count * 8
)

confidence_score = max(
    0,
    min(100, confidence_score)
)


# ================================================================
# 23. OVERALL INTERVIEW SCORE
# ================================================================

overall_score = (

    communication_score * 0.25

    +

    technical_score * 0.25

    +

    soft_skill_score * 0.15

    +

    confidence_score * 0.20

    +

    relevance_score * 0.15

)

overall_score = round(
    overall_score
)


# ================================================================
# 24. PERFORMANCE LEVEL
# ================================================================

if overall_score >= 85:

    performance = "Excellent ⭐⭐⭐⭐⭐"

elif overall_score >= 70:

    performance = "Very Good ⭐⭐⭐⭐"

elif overall_score >= 55:

    performance = "Good ⭐⭐⭐"

elif overall_score >= 40:

    performance = "Needs Improvement ⭐⭐"

else:

    performance = "Poor ⭐"


# ================================================================
# 25. STRENGTHS
# ================================================================

strengths = []

if communication_score >= 70:

    strengths.append(
        "Good communication structure"
    )

if technical_score >= 60:

    strengths.append(
        "Good technical keyword coverage"
    )

if soft_skill_score >= 60:

    strengths.append(
        "Good soft-skill awareness"
    )

if confidence_score >= 70:

    strengths.append(
        "Shows confidence"
    )

if total_fillers <= 2:

    strengths.append(
        "Uses few filler words"
    )

if relevance_score >= 60:

    strengths.append(
        "Answer is reasonably relevant"
    )


if not strengths:

    strengths.append(
        "Candidate has potential to improve with practice"
    )


# ================================================================
# 26. IMPROVEMENT AREAS
# ================================================================

improvements = []

if communication_score < 60:

    improvements.append(
        "Improve answer clarity and communication"
    )

if technical_score < 50:

    improvements.append(
        "Include more relevant technical concepts"
    )

if soft_skill_score < 50:

    improvements.append(
        "Give examples of teamwork and problem-solving"
    )

if confidence_score < 60:

    improvements.append(
        "Use more confident language"
    )

if total_fillers >= 3:

    improvements.append(
        "Reduce filler words such as 'um', 'like', and 'actually'"
    )

if uncertainty_count >= 2:

    improvements.append(
        "Avoid uncertain phrases such as 'maybe' and 'I think'"
    )

if relevance_score < 50:

    improvements.append(
        "Keep the answer more closely related to the question"
    )

if word_count < 30:

    improvements.append(
        "Provide a more detailed answer with an example"
    )


if not improvements:

    improvements.append(
        "Continue practicing to maintain consistency"
    )


# ================================================================
# 27. FINAL REPORT
# ================================================================

print("\n")
print("=" * 75)
print("                    📊 INTERVIEW REPORT")
print("=" * 75)


print("\n👤 INTERVIEW QUESTION")
print("-" * 75)
print(question)


print("\n💬 ANSWER")
print("-" * 75)
print(clean_answer)


print("\n😊 SENTIMENT ANALYSIS")
print("-" * 75)

print("Sentiment       :", sentiment)
print(
    "Confidence      :",
    f"{sentiment_confidence:.2f}%"
)


print("\n🎭 EMOTION ANALYSIS")
print("-" * 75)

print("Emotion         :", emotion)
print(
    "Confidence      :",
    f"{emotion_confidence:.2f}%"
)


print("\n🗣️ COMMUNICATION ANALYSIS")
print("-" * 75)

print(
    "Communication Score:",
    f"{communication_score}/100"
)

print(
    "Word Count         :",
    word_count
)

print(
    "Sentence Count     :",
    sentence_count
)

print(
    "Average Sentence Length:",
    f"{average_sentence_length:.2f} words"
)

print(
    "Filler Words        :",
    total_fillers
)


print("\n🛠️ TECHNICAL SKILLS")
print("-" * 75)

print(
    "Technical Score:",
    f"{technical_score}/100"
)

if detected_technical:

    print(
        "Detected Skills:",
        ", ".join(detected_technical)
    )

else:

    print(
        "Detected Skills: No major technical keywords found"
    )


print("\n🤝 SOFT SKILLS")
print("-" * 75)

print(
    "Soft Skill Score:",
    f"{soft_skill_score}/100"
)

if detected_soft_skills:

    print(
        "Detected:",
        ", ".join(detected_soft_skills)
    )

else:

    print(
        "Detected: No major soft-skill keywords found"
    )


print("\n💪 CONFIDENCE ANALYSIS")
print("-" * 75)

print(
    "Confidence Score:",
    f"{confidence_score}/100"
)

print(
    "Positive Confidence Indicators:",
    confidence_count
)

print(
    "Uncertain Indicators:",
    uncertainty_count
)


print("\n🎯 ANSWER RELEVANCE")
print("-" * 75)

print(
    "Relevance Score:",
    f"{relevance_score:.2f}/100"
)


print("\n📋 ANSWER STRUCTURE")
print("-" * 75)

if detected_structure:

    print(
        "Detected:",
        ", ".join(detected_structure)
    )

else:

    print(
        "No clear Situation → Action → Result structure detected."
    )


print("\n🎤 SPEECH ANALYSIS")
print("-" * 75)

if choice == "2":

    if audio_duration:

        print(
            "Audio Duration:",
            f"{audio_duration:.2f} seconds"
        )

    if speech_rate:

        print(
            "Speech Rate:",
            f"{speech_rate:.2f} words/minute"
        )

        if speech_rate < 100:

            print(
                "Speaking Speed: Slow"
            )

        elif speech_rate <= 160:

            print(
                "Speaking Speed: Normal ✅"
            )

        else:

            print(
                "Speaking Speed: Fast"
            )

else:

    print(
        "Speech analysis not selected."
    )


print("\n⭐ OVERALL INTERVIEW SCORE")
print("-" * 75)

print(
    "Overall Score:",
    f"{overall_score}/100"
)

print(
    "Performance :",
    performance
)


print("\n💚 STRENGTHS")
print("-" * 75)

for strength in strengths:

    print("✅", strength)


print("\n⚠️ AREAS FOR IMPROVEMENT")
print("-" * 75)

for improvement in improvements:

    print("•", improvement)


print("\n🚀 FINAL RECOMMENDATION")
print("-" * 75)

if overall_score >= 85:

    print(
        "Excellent performance. The candidate appears "
        "well prepared for the interview."
    )

elif overall_score >= 70:

    print(
        "Very good performance. Focus on polishing "
        "technical answers and communication."
    )

elif overall_score >= 55:

    print(
        "Good potential. More interview practice "
        "will improve confidence and answer quality."
    )

else:

    print(
        "More preparation is recommended. Practice "
        "technical questions, communication, and confidence."
    )


print("\n" + "=" * 75)
print("             🎤 INTERVIEW ANALYSIS COMPLETED")
print("=" * 75)

print("""
⚠️ PROJECT NOTE:

This application provides an AI-assisted interview evaluation.
The scores are indicators, not a professional hiring decision.

For a final-year project, the system can be extended with:
• Resume analysis
• Multiple interview questions
• Voice emotion analysis
• Facial expression analysis
• Question generation
• Interview history
• Candidate comparison
• PDF report generation
• Web dashboard
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 54.7 MB/s eta 0:00:00
             🎤 AI INTERVIEW ANALYZER
                 TEXT + SPEECH ANALYSIS

⏳ Loading AI models...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ AI models loaded successfully!

SELECT INTERVIEW INPUT

1. Text Answer
2. Speech / Audio Answer

Enter 1 or 2: Tell me about yourself and your technical skills.

❌ Invalid choice.


SystemExit: 

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
